# 🔮 Notebook 5 — Hypothetical Scenario Engine
**Formula 1 ML Analytics Project**

**Question answered**: *Can driver X win in team Y? (e.g., Vettel in Alfa Romeo)*

This notebook demonstrates the 20-parameter hypothetical scenario engine that:
1. Extracts a driver's skill profile (independent of car)
2. Extracts a team's car performance profile
3. Combines them under configurable conditions
4. Runs 1,000 Monte Carlo season simulations to produce a points distribution

**20 configurable parameters**: driver, team, year, track, grid override, teammate, weather,
pit stops, reliability factor, season round, driver age, car development rate, tire strategy,
starting points, confidence factor, regulation era, races to simulate, teammate skill,
race incidents, home race.


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
DATA_PATH = "../data/processed/"
MODEL_PATH = "../models/"

feat_file = os.path.join(DATA_PATH, "featured_df.csv")
if not os.path.exists(feat_file):
    print("⚠️  Run notebooks 01 and 02 first.")
else:
    featured_df = pd.read_csv(feat_file, low_memory=False)
    print(f"✅ Loaded featured_df: {featured_df.shape}")

# Load race winner model if available
try:
    import joblib
    model_path = os.path.join(MODEL_PATH, "race_winner_model.pkl")
    race_model = joblib.load(model_path) if os.path.exists(model_path) else None
    print(f"Race model: {'✅ loaded' if race_model else '⚠️  not found (heuristic mode)'}")
except Exception as e:
    race_model = None
    print(f"Race model: ⚠️  not loaded ({e})")


In [ ]:
from src.hypothetical_engine import (
    get_driver_skill_profile,
    get_team_car_profile,
    hypothetical_scenario
)

# -----------------------------------------------------------------------
# EXAMPLE 1: Sebastian Vettel in Alfa Romeo (2022)
# -----------------------------------------------------------------------
print("="*60)
print("🔮 SCENARIO 1: Sebastian Vettel in Alfa Romeo (2022)")
print("="*60)
driver_profile = get_driver_skill_profile(featured_df, driver="Sebastian Vettel", year=2022)
team_profile   = get_team_car_profile(featured_df, team="Alfa Romeo", year=2022)

print("\nDriver skill profile (Vettel):")
for k, v in driver_profile.items():
    print(f"  {k}: {v}")
print("\nTeam car profile (Alfa Romeo 2022):")
for k, v in team_profile.items():
    print(f"  {k}: {v}")


In [ ]:
# Run the full hypothetical scenario
result1 = hypothetical_scenario(
    featured_df      = featured_df,
    driver           = "Sebastian Vettel",
    team             = "Alfa Romeo",
    year             = 2022,
    reliability_factor = 0.88,          # Alfa's typical 2022 reliability
    regulation_era   = "ground_effect", # 2022 regulations
    race_incidents   = True,
    teammate_skill_level = "midfield",
    n_monte_carlo    = 1000,
    model            = race_model
)

print("\n📊 RESULTS: Vettel in Alfa Romeo 2022 — Full Season Simulation")
print("="*60)
print(f"  Predicted season points:   {result1.get('season_points_estimate', 'N/A'):.1f}")
print(f"  Predicted wins:            {result1.get('season_wins_estimate', 'N/A'):.1f}")
print(f"  Predicted podiums:         {result1.get('season_podiums_estimate', 'N/A'):.1f}")
print(f"  Championship position est: {result1.get('championship_position_estimate', 'N/A')}")
ci = result1.get('confidence_interval', (None,None))
print(f"  90% Confidence interval:   [{ci[0]:.0f}, {ci[1]:.0f}] points")


In [ ]:
# Monte Carlo distribution
mc = result1.get('monte_carlo_results', [])
if mc:
    from src.visualizations import plot_monte_carlo_distribution
    plot_monte_carlo_distribution(mc, driver="Sebastian Vettel", team="Alfa Romeo", year=2022)


In [ ]:
# -----------------------------------------------------------------------
# EXAMPLE 2: Lewis Hamilton in Haas (2023) — "Can Hamilton win in a weak car?"
# -----------------------------------------------------------------------
print("="*60)
print("🔮 SCENARIO 2: Lewis Hamilton in Haas F1 Team (2023)")
print("="*60)
result2 = hypothetical_scenario(
    featured_df      = featured_df,
    driver           = "Lewis Hamilton",
    team             = "Haas F1 Team",
    year             = 2023,
    reliability_factor = 0.85,
    regulation_era   = "ground_effect",
    race_incidents   = True,
    teammate_skill_level = "midfield",
    n_monte_carlo    = 1000,
    model            = race_model
)
print(f"\n  Predicted season points:   {result2.get('season_points_estimate','N/A'):.1f}")
print(f"  Predicted wins:            {result2.get('season_wins_estimate','N/A'):.1f}")
print(f"  Championship position est: {result2.get('championship_position_estimate','N/A')}")
ci2 = result2.get('confidence_interval',(None,None))
print(f"  90% CI:                    [{ci2[0]:.0f}, {ci2[1]:.0f}] points")

mc2 = result2.get('monte_carlo_results', [])
if mc2:
    plot_monte_carlo_distribution(mc2, driver="Lewis Hamilton", team="Haas F1 Team", year=2023)


In [ ]:
# -----------------------------------------------------------------------
# EXAMPLE 3: Max Verstappen in 2014 Mercedes (Hybrid era at peak)
# -----------------------------------------------------------------------
print("="*60)
print("🔮 SCENARIO 3: Max Verstappen in Mercedes (2014 — V6 Hybrid)")
print("="*60)
result3 = hypothetical_scenario(
    featured_df      = featured_df,
    driver           = "Max Verstappen",
    team             = "Mercedes",
    year             = 2014,
    regulation_era   = "v6_hybrid",
    weather          = "dry",
    reliability_factor = 0.95,
    race_incidents   = True,
    driver_confidence_factor = 0.7,
    n_monte_carlo    = 1000,
    model            = race_model
)
print(f"\n  Predicted season points:   {result3.get('season_points_estimate','N/A'):.1f}")
print(f"  Predicted wins:            {result3.get('season_wins_estimate','N/A'):.1f}")
print(f"  Championship position est: {result3.get('championship_position_estimate','N/A')}")


In [ ]:
# -----------------------------------------------------------------------
# Single race prediction: Alain Prost in Monaco 1984
# -----------------------------------------------------------------------
print("="*60)
print("🔮 SCENARIO 4: Alain Prost — Monaco Grand Prix 1984")
print("="*60)
result4 = hypothetical_scenario(
    featured_df      = featured_df,
    driver           = "Alain Prost",
    team             = "McLaren",
    year             = 1984,
    track            = "Monaco Grand Prix",
    weather          = "wet",     # famous wet Monaco 1984 race
    regulation_era   = "v8",
    grid_position_override = 1,
    race_incidents   = True,
    n_monte_carlo    = 100,
    model            = race_model
)
print(f"  Win probability:   {result4.get('predicted_win_probability', 'N/A')}")
print(f"  Predicted position: {result4.get('predicted_position', 'N/A')}")
